# INSTRUCTIONS: 
Please execute the cells in the notebook as isntructed. You will have to complete some TODO sections in order to proceed further and other cells you will just have to RUN.

## 1. Setup the environment and define utility function (RUN only)

You just need to run them as is and no need to read them!

In [ ]:
!pip install transformers
!pip install Jinja2==3.1.6
!pip install accelerate

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", 
                                             device_map="auto",
                                             dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
import uuid
import time
from IPython.display import display, HTML, Javascript
import html as html_lib

# --- Model generation utility ---
def generate_model_response(prompt):
    """
    Runs the model on a given prompt and returns the response text.
    """
    start_time = time.time()
    inputs = tokenizer.apply_chat_template(
        prompt,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.8)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    elapsed = time.time() - start_time
    return response, elapsed

# --- Display utility ---
def display_model_interaction(prompt):
    """
    Displays the prompt and model response in a styled table,
    with a loading indicator while the model generates text.
    """
    uid = str(uuid.uuid4()).replace("-", "")

    # HTML scaffold (before generation)
    html = f"""
    <style>
    .prompt-table {{
        border-collapse: collapse;
        width: 100%;
        margin: 12px 0;
        font-family: 'Segoe UI', sans-serif;
    }}
    .prompt-table th, .prompt-table td {{
        border: 1px solid #ccc;
        padding: 10px;
        vertical-align: top;
    }}
    .prompt-table th {{
        background-color: #f2f2f2;
        width: 20%;
    }}
    .loading {{
        color: #888;
        font-style: italic;
        animation: pulse 1.5s infinite;
    }}
    @keyframes pulse {{
        0% {{ opacity: 0.3; }}
        50% {{ opacity: 1; }}
        100% {{ opacity: 0.3; }}
    }}
    </style>
    <table class="prompt-table">
        <tr><th>Prompt</th><td>{prompt}</td></tr>
        <tr><th>Model Response</th><td id="response_{uid}">
            <span class="loading">⏳ Generating response...</span>
        </td></tr>
    </table>
    """

    # Display initial table
    display(HTML(html))

    # --- Call the model separately ---
    response_text, elapsed = generate_model_response(prompt)
    # print(response_text)
    # Escape special chars for HTML display
    safe_response = html_lib.escape(response_text.strip())

    # --- Inject response dynamically ---
    js = Javascript(f"""
        document.getElementById("response_{uid}").innerHTML =
            `<pre style="white-space: pre-wrap;">{safe_response}</pre>
             <div style='color:#666; font-size:90%; margin-top:4px;'>⏱ Generated in {elapsed:.2f} seconds</div>`;
    """)
    display(js)

def display_sample(question, answer):
    display(HTML(
        f"""<p>Question/answer 0:</p>
        <strong>Question: </strong>{question}
        <p><strong>Answer: </strong>{answer}</p>
        """
    ))retur

## Tree of Thought

Before diving into the code, let’s recall what Tree of Thought (ToT) means.
Instead of letting an LLM think in one straight line (like standard chain-of-thought), ToT lets it branch out 🌳 — exploring several possible reasoning paths (called beams), checking which one looks most promising, and then expanding that path further.
Think of it as the model brainstorming multiple ideas, comparing them, and then following the best one step by step.

Each section below walks through one stage of this reasoning process — from setup 🛠️ to proposal 💡 to composition

### 1) Prompts for ToT instruction, propose & score (RUN only)

We set global constants such as the beam width (B) and depth (D), which control how wide and deep the reasoning tree grows.

🧮 Wider → more ideas explored; deeper → longer reasoning chains.


In [ ]:
from typing import List, Tuple
import random

# --- Config ---
SEED = 42
random.seed(SEED)

# Beam search / ToT parameters
B = 2   # branching factor: candidates per step
D = 2   # tree depth: outline beams (e.g., beginning / middle / end)

We keep prompts tiny and task-specific. These templates encode the ToT task (e.g., story outline, problem solving) and instruct the model on how to expand reasoning branches:

- **Task instruction**: tells the agent what the downstream task is.
- **Propose prompt**: asks for the *next outline beam* (one or two sentences).
- **Score prompt**: asks for a *single numeric score* 0–10 for coherence, creativity, and fit.

[OPTIONAL] Feel free to play around with these prompts and see if the results change!

In [ ]:
TASK_INSTRUCTION = (
    "Write a vivid, 100 word short story for a general audience. "
    "It should feature: a surprising twist, strong imagery, and an emotionally satisfying resolution. "
    "Avoid proper names and instead use descriptive nouns."
)

PROPOSE_PROMPT_TEMPLATE = """
You are helping outline a short story. The global writing brief is:

{task}

Given the current outline (beams so far):
{outline}

Propose ONE next beam in 1-2 sentences that continues the outline naturally.
Keep it concrete and evocative. Do not repeat earlier beams.
Return ONLY the next beam, no explanations.
""".strip()

SCORE_PROMPT_TEMPLATE = """
You are evaluating a candidate *next beam* for a short-story outline.

Global brief:
{task}

beams so far:
{outline}

Candidate next beam:
{candidate_prompt}

Score the candidate from 0 to 10 for:
- Coherence with beams so far
- Creativity (fresh but plausible)
- Fitness for the brief (tone, twist potential, resolution potential)

Return ONLY a single integer 0-10 (no words).
""".strip()


In the prompts above, you will see there are certain placeholders ({task}, {outline}, etc.). In order to understand how we fill out the placeholders in a prompt, check the following code.

In [ ]:
prompt = PROPOSE_PROMPT_TEMPLATE.format(task=TASK_INSTRUCTION, outline="Testing")
print(prompt)

#### TODO: Try it yourself!

Now try creating the SCORE_PROMPT_TEMPLATE with some dummy text. Remember, you have to send as many texts as the number of placeholders in the prompt.

In [ ]:
#your code here

This was just for you to understand how to fill out a template in Python. In the following code, you have to put these skills to use!

### 2) ToT functions (propose → score → select)

- `propose_next_beams`: ask the LLM for `B` candidate beams.  
- `score_candidates`: ask the LLM to return **just an integer** score per beam.  
- Simple beam search keeps the top `B` partial outlines at each depth.


[IMPORTANT] Before you run the following functions, please read the following: 

You will see that the message to the model is being sent in a particular format where you clearly define the role and the corresponding content. This is called the Chat Template. The message sent to the model is always an array of such dictionaries where inside each Dict you define:

- **'role'**: user or assistant
- **'content'**: the actual prompt or message

### TODO: 
There are 3 tasks for you in the following cell. Please complete them to move forward!

In [ ]:
def propose_next_beams(outline_beams: List[str], b: int, client) -> List[str]:
    outline_text = "\n- ".join([""] + outline_beams) if outline_beams else "(none yet)"
    prompt = PROPOSE_PROMPT_TEMPLATE.format(task=TASK_INSTRUCTION, outline=outline_text)
    message = [{"role": "user", "content": prompt}]
    #TODO: call the model with this message and then return the response
    pass

def score_candidates(outline_beams: List[str], candidate_prompts: List[str], client) -> List[Tuple[str, float]]:
    scored = []
    outline_text = "\n- ".join([""] + outline_beams) if outline_beams else "(none yet)"
    for cand_prompt in candidate_prompts:
        #TODO: create the prompt using SCORE_PROMPT_TEMPLATE and .format()
        message = [] # TODO: create the message to be sent to the model
        try:
            resp = generate_model_response(message)[0].strip()
            # Extract a single integer 0-10; be robust to stray text
            digits = ''.join(ch for ch in resp if ch.isdigit())
            score = float(digits) if digits else 0.0
            score = max(0.0, min(10.0, score))
        except Exception as e:
            score = 0.0
        scored.append((cand_prompt, score))
    return scored


### 3) TODO: Beam search over outline beams

This is the core of the Tree-of-Thought algorithm 🌳

It loops through multiple steps of:
1.	Propose candidate thoughts using the propose function above
2.	Evaluate and select the top `B` outlines using the score function above
3.	Expand further
until it builds a complete reasoning outline.

Your task is to call the respective functions in the first 2 steps (marked as TODO below).

In [ ]:
def beam_search_outline(b: int = B, d: int = D) -> List[str]:
    # Each state is (beams, avg_score)
    beams: List[Tuple[List[str], float]] = [([], 0.0)]
    for step in range(1, d + 1):
        new_beams: List[Tuple[List[str], float]] = []
        for beams, avg_score in beams:
            candidate_prompts = # TODO: Propose next candidates
            scored = # TODO: Score the candidates
            for cand, s in scored:
                new_beams.append((beams + [cand], (avg_score * (step-1) + s) / step))
        # Keep the top B beams
        new_beams.sort(key=lambda x: x[1], reverse=True)
        beams = new_beams[:b]
        print(f"Step {step}: top avg score = {beams[0][1]:.2f}")
    return beams[0][0]  # best outline beams


### 4) TODO: Compose final story from the best outline

Once we have the outline beams, we ask the LLM to write the full story.
Complete the TODO parts here to proceed further

In [ ]:
COMPOSE_PROMPT_TEMPLATE = """
Using the outline below, write the final story (100 words). Follow the brief closely.
Write in a single block of prose.

Brief:
{task}

Outline beams:
- {beams}

Begin the story now.
""".strip()

def compose_story_from_outline(beams: List[str]) -> str:
    beams_text = "\n- ".join(beams)
    # TODO: create the prompt using PROPOSE_PROMPT_TEMPLATE and .format()
    # TODO: create the message with this prompt
    # TODO: call the display utility function


### 5) Run the full ToT pipeline (RUN only)

In [ ]:
best_outline = beam_search_outline(b=B, d=D)
print("\nBest outline beams:")
for i, beam in enumerate(best_outline, 1):
    print(f"{i}. {beam}")

print("\n--- Final Story ---\n")
compose_story_from_outline(best_outline)

### Baseline: direct 0-shot story (RUN only)

Now let's just prompt the model with the task instruction (zero-shot prompting).

In [ ]:
BASELINE_PROMPT = """
{task}

Write the story now (100 words) in one block.
""".strip()

def baseline_story() -> str:
    prompt = BASELINE_PROMPT.format(task=TASK_INSTRUCTION)
    message = [{"role": "user", "content": prompt}]
    display_model_interaction(model, tokenizer, message)

baseline_story()

# Subquestion Decomposition

### Basic Approach:

RECAP

Sometimes, a big reasoning problem is too hard to solve in one go 🧠💥.
Instead of attacking it directly, we decompose it into smaller, simpler sub-questions — each one focusing on a single aspect of the problem.
The model then answers each sub-question separately and combines the results for a clearer, more accurate final answer.

This approach mimics how humans tackle complex tasks: think → split → solve → combine 🔄

#### Load dataset
Let's prompt the model to generate subquestions and answer them to reach the final answer. For this, we first load the socratic version of the same dataset provided by the creators.

In [ ]:
ds_socratic = load_dataset("openai/gsm8k", "socratic")
train_split_socratic = ds_socratic['train']
test_split_socratic = ds_socratic['test']

print(f"Train split size: {len(train_split_socratic)}")
print(f"Test split size: {len(test_split_socratic)}")

In [ ]:
display_sample(train_split_socratic[0]['question'], train_split_socratic[0]['answer'])

#### Few-shot (RUN only)

Now, let's try this approach using Few-shot prompting first. We will use some samples from the train set as exemplars and then invoke the model to solve a question from the test split.

In [ ]:
few_shot_prompt_socratic = []

for i in range(1):
    s = train_split_socratic[i]
    q = str(s["question"]).strip()
    a = str(s["answer"]).strip()
    few_shot_prompt_socratic.append(f"Q: {q}\nA: {a}")

test_q = str(test_split_socratic[0]["question"]).strip()
blocks = []
instruction = "Here is an example on how to solve the question:"
blocks.append(instruction.strip())
blocks.extend(few_shot_prompt_socratic)
blocks.append(f"Please follow the same approach to answer the following question:\nQ: {test_q}\n")

few_shot_soc = "\n\n".join(blocks)
print(few_shot_soc)

In [ ]:
message = [{"role": "user", "content": few_shot_soc}]
display_model_interaction(message)

#### TODO: 0-shot

Create a prompt where you tell the model to follow this technique. TIP: instructions should be short, simple and to the point.

In [ ]:
#TODO: add your prompt here for zero-shot
#TODO: create the message to be sent to the model
#TODO: call the display utility

### Plan-and-Solve

The Plan-and-Solve technique teaches the model to separate planning from execution — just like how humans first sketch an approach before diving into work.

🧠 How it works:
1.	Plan Phase 🗺️ — The model outlines a clear sequence of steps or a rough plan for solving the problem.
“First, identify what’s asked → then gather given info → compute step by step.”
2.	Solve Phase 🧮 — The model follows its own plan systematically to produce the final answer.


Below you will find 2 few-shot examples that have been prepared for this prompting technique. Again, here we will use few-shot and 0-shot prompting.

#### Few-shot (RUN only)

In [ ]:
few_shot_PS = f"""
Q: James decides to run 3 sprints 3 times a week. He runs 60 meters each sprint. How many total meters does he run a week?
A: Let's first understand the problem, extract relevant variables and their corresponding numerals, and make a complete plan.Then, let's carry out the plan, calculate intermediate variables (pay attention to correct numerical calculation and commonsense), solve the problem step by step, and show the answer.
Output:
Given:
James runs 3 sprints 3 times a week.
Each sprint is 60 meters.
Plan:
We need to calculate the total meters run by James in a week.
Calculation:
Total number of sprints run by James in a week = 3 sprints x 3 times = 9 sprints
Total meters run by James in a week = 9 sprints x 60 meters = 540 meters
Answer:
James runs 540 meters in a week.

Q: In a dance class of 20 students, 20% enrolled in contemporary dance, 25% of the remaining enrolled in jazz dance, and the rest enrolled in hip-hop dance. What percentage of the entire students enrolled in hip-hop dance?
A: Let's first understand the problem, extract relevant variables and their corresponding numerals, and make a complete plan.Then, let's carry out the plan, calculate intermediate variables (pay attention to correct numerical calculation and commonsense), solve the problem step by step, and show the answer.
Output:
Given:
Total number of students = 20
Percentage of students enrolled in contemporary dance = 20%
Percentage of students enrolled in jazz dance = 25%
Plan:
1. Calculate the number of students enrolled in contemporary dance.
2. Calculate the number of students remaining after subtracting the number of students enrolled in
contemporary dance.
3. Calculate the number of students enrolled in jazz dance.
4. Calculate the number of students enrolled in hip-hop dance.
5. Calculate the percentage of students enrolled in hip-hop dance.
Calculation:
1. Number of students enrolled in contemporary dance = 20% of 20 = 20% * 20 = 4
2. Number of students remaining after subtracting the number of students enrolled in contemporary
dance = 20 - 4 = 16
3. Number of students enrolled in jazz dance = 25% of 16 = 25% * 16 = 4
4. Number of students enrolled in hip-hop dance = 16 - 4 = 12
5. Percentage of students enrolled in hip-hop dance = 12/20 * 100% = 60%
Answer:
60% of the entire students enrolled in hip-hop dance.

Q: {test_split_socratic[0]['question']}
A:
"""

In [ ]:
message = [{"role": "user", "content": few_shot_PS}]
display_model_interaction(model, tokenizer, message)

#### TODO: 0-shot

Create a prompt with instructions to follow this technique

In [ ]:
#TODO: add your prompt here for zero-shot
#TODO: create the message to be sent to the model
#TODO: call the display utility

# KEY TAKEAWAYS

Woohoo!! You explored quite a few interesting prompting techniques. Now let's make a note of some imporatnt observations:

1. You saw how zero-shot prompting reduces a lot of work by avoiding adding examples. There might be cases where you don't have any precedents and can only describe the task - that's where zero-shot reasoning comes to your rescue!
2. All these reasoning techniques increase the model accuracy and also give us an insight of how it approaches the problem.
3. Hence, like humans, LLMs reason better when allowed to explore, refine or decompose.